# Human or AI? · Group 21
## Reproduce the selected 75/25 blend
A gradient-boosted style and language-model classifier and a word TF-IDF model provide the v2 score. Fine-tuned DistilRoBERTa supplies the other component. Their standardized scores receive weights of 0.75 and 0.25.

On 2,000 held-out originals the blend reached **0.996868 AUROC**. It scored **0.978895** when case and punctuation were removed. The reported public Kaggle AUROC was **0.95934**.

Select a T4 GPU runtime. Place the four competition CSVs in `MyDrive/human-or-ai/data`. The experiment sources live in the four notebooks beside this launcher on GitHub.

## 1. Open the team code and data
The first cell downloads the submission branch if it is absent. Outputs go to a separate folder in Drive so the historical experiment files stay available.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
checkout = Path('/content/group21-course-submission')
if not checkout.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'submission/group-21',
                    'https://github.com/g-maanasi/human-or-ai-submissions-group.git',
                    str(checkout)], check=True)
else:
    subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', 'origin', 'submission/group-21'], check=True)
project = checkout / 'submissions/group-21'
data = Path('/content/drive/MyDrive/human-or-ai/data')
output = data.parent / 'outputs_group21_reproduction'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(project / 'requirements.txt')], check=True)
print('Code revision:', subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Train and evaluate
The pipeline recreates the stratified split, fits the baselines, stress-tests v2, and fine-tunes the transformer. The training diagnostic also scores the 8,000 original fit passages with the validation-stage blend. The final test models use all 10,000 labels. A full run requires a GPU and can take substantially longer than scoring saved probes.

In [ ]:
subprocess.run([sys.executable, str(project / 'train.py'), '--data-dir', str(data),
                '--out-dir', str(output)], cwd=project,
               env={**os.environ, 'RUN_TRAIN_DIAGNOSTIC': '1'}, check=True)

## 3. Inspect held-out performance
Compare models on identical edited strings. The diagnostics report length-group AUROC and the five most confident errors in each direction at a 0.5 cutoff.

In [ ]:
import pandas as pd
run = output / 'distilroberta_v1'
display(pd.read_csv(run / 'paired_blend_auc.csv'))
subprocess.run([sys.executable, str(project / 'distilroberta/diagnostics.py'), '--train', str(data / 'train.csv'),
                '--v2-probes', str(run / 'v2_validation_probes.csv.gz'),
                '--transformer-probes', str(run / 'validation_probes.csv.gz')], check=True)

## 4. Predict on the test set and download the CSV
The file contains one AI-authorship probability for each test ID in Kaggle sample order.

In [ ]:
from google.colab import files
subprocess.run([sys.executable, str(project / 'predict.py'), '--data-dir', str(data),
                '--out-dir', str(output)], cwd=project, check=True)
path = output / 'submission.csv'
predictions = pd.read_csv(path)
sample = pd.read_csv(data / 'sample_submission.csv')
assert predictions.columns.tolist() == ['id', 'label']
assert len(predictions) == 3000 and predictions.id.is_unique
assert predictions.id.equals(sample.id) and predictions.label.between(0, 1).all()
files.download(str(path))